# Comparación controlada: ¿aportan los embeddings del autoencoder?

La comparación previa entre `baseline.ipynb` (solo handcrafted) y `lstm_ae.ipynb` (handcrafted+embeddings) **no era válida**: cada notebook usaba un split distinto —el baseline con `GroupShuffleSplit(test=0.20)` y el pipeline del AE con `split_subjects(test_frac=0.15)`—, así que los *test sets* tenían sujetos distintos (47427 vs 31465 épocas). Cualquier diferencia de Kappa quedaba confundida con el cambio de partición.

Este cuaderno rehace la comparación **apples-to-apples**: **un solo split** (el mismo mecanismo que `baseline.ipynb`) y **exactamente los mismos train/val/test rows** para los tres modelos, que difieren **solo** en qué columnas de features usan:

- **FEAT_HC** — las 122 features handcrafted del baseline.
- **FEAT_EMB** — solo las 32 dimensiones aprendidas por el autoencoder (`emb_0..emb_31`).
- **FEAT_ALL** — las 154 (handcrafted + embeddings).

Misma `XGBClassifier` (mismos hiperparámetros, early stopping, `eval_metric`) y misma semilla en los tres. Así el aporte de los embeddings queda aislado en dos números: los deltas `EMB - HC` y `ALL - HC`.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import numpy as np
import pandas as pd

## 1. Carga y merge por `(subject, night, epoch)`

Merge **inner** de las features handcrafted (`epoch_features.csv`) con los embeddings del AE (`ae_embeddings.parquet`). Verificamos que no se pierdan filas, que no haya claves duplicadas y que las `emb_*` no traigan NaN.

In [ ]:
feat = pd.read_csv("../data_extraction/epoch_features.csv")
emb = pd.read_parquet("../data_extraction/ae_embeddings.parquet")

meta_cols = ['subject', 'night', 'epoch', 'label', 'dreem']
hc_cols = [c for c in feat.columns if c not in meta_cols]
emb_cols = [c for c in emb.columns if c.startswith('emb_')]

# label/dreem se toman de feat para no duplicarlos; del emb solo traemos claves + embeddings
df = feat.merge(emb[['subject', 'night', 'epoch'] + emb_cols],
                on=['subject', 'night', 'epoch'], how='inner')

print(f"handcrafted : {len(hc_cols)} features")
print(f"embeddings  : {len(emb_cols)} features")
print(f"filas -> feat: {len(feat)}  emb: {len(emb)}  merge(inner): {len(df)}")
print(f"claves duplicadas en merge: {df.duplicated(['subject','night','epoch']).sum()}")
print(f"NaN en columnas emb_*: {int(df[emb_cols].isna().sum().sum())}")
assert len(df) == len(feat) == len(emb), "el merge perdió filas: revisar claves antes de seguir"
assert df.duplicated(['subject','night','epoch']).sum() == 0
assert int(df[emb_cols].isna().sum().sum()) == 0

## 2. Split único (idéntico a `baseline.ipynb`)

`GroupShuffleSplit` agrupado por `subject` (evita fuga entre épocas correlacionadas del mismo paciente): dev 80% / test 20%, y dentro de dev train 80% / val 20%. Mismo `random_state=42` que el baseline. Descartamos `label==5` (Unknown). Guardamos las posiciones de fila de cada partición: **los tres modelos usan exactamente estos mismos rows**.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

# descartamos Unknown y reindexamos: las posiciones 0..N-1 son la referencia común
df = df[df['label'] != 5].reset_index(drop=True)
groups = df['subject'].values
y_all = df['label'].values

# dev / test agrupado por paciente (mismo seed que baseline)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
dev_pos, test_pos = next(gss.split(df, y_all, groups))

# dentro de dev: train / val, también por paciente (índices relativos a dev -> a absolutos)
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_rel, val_rel = next(gss_val.split(dev_pos, y_all[dev_pos], groups[dev_pos]))
train_pos = dev_pos[tr_rel]
val_pos = dev_pos[val_rel]

y_train = y_all[train_pos]
y_val = y_all[val_pos]
y_test = y_all[test_pos]
dreem_test = df['dreem'].values[test_pos]

subj_train = sorted(np.unique(groups[train_pos]).tolist())
subj_val = sorted(np.unique(groups[val_pos]).tolist())
subj_test = sorted(np.unique(groups[test_pos]).tolist())

print(f"épocas    -> train: {len(train_pos)}  val: {len(val_pos)}  test: {len(test_pos)}")
print(f"pacientes -> train: {len(subj_train)}  val: {len(subj_val)}  test: {len(subj_test)}")
print(f"\nsujetos TRAIN: {subj_train}")
print(f"sujetos VAL  : {subj_val}")
print(f"sujetos TEST : {subj_test}")

# sanity: particiones disjuntas por sujeto
assert set(subj_train).isdisjoint(subj_val)
assert set(subj_train).isdisjoint(subj_test)
assert set(subj_val).isdisjoint(subj_test)

épocas    -> train: 123495  val: 37566  test: 47427
pacientes -> train: 29  val: 8  test: 10

sujetos TRAIN: [0, 1, 2, 6, 9, 11, 13, 14, 15, 17, 18, 20, 22, 30, 34, 35, 36, 42, 43, 44, 45, 50, 51, 55, 60, 62, 63, 65, 68]
sujetos VAL  : [8, 10, 19, 32, 47, 49, 53, 66]
sujetos TEST : [7, 16, 31, 38, 39, 40, 41, 52, 56, 64]


## 3. Tres conjuntos de features, misma config de XGBoost

Los tres `XGBClassifier` comparten hiperparámetros, `early_stopping_rounds`, `eval_metric` y semilla (idénticos a `baseline.ipynb`); solo cambia la lista de columnas. Todos se entrenan sobre `train_pos`, hacen early stopping sobre `val_pos` y se evalúan sobre `test_pos`.

In [4]:
from xgboost import XGBClassifier

FEATURE_SETS = {
    'HC': hc_cols,                 # 122 handcrafted
    'EMB': emb_cols,               # 32 embeddings del AE
    'ALL': hc_cols + emb_cols,     # 154
}

def fit_xgb(cols):
    X = df[cols]
    clf = XGBClassifier(
        n_estimators=600,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='mlogloss',
        early_stopping_rounds=30,
        n_jobs=-1,
        random_state=42,
    )
    clf.fit(X.iloc[train_pos], y_train,
            eval_set=[(X.iloc[val_pos], y_val)], verbose=False)
    return clf

models = {}
for name, cols in FEATURE_SETS.items():
    models[name] = fit_xgb(cols)
    print(f"{name:3s} ({len(cols):3d} feats) -> mejor iteración (early stopping): {models[name].best_iteration}")

HC  (122 feats) -> mejor iteración (early stopping): 159
EMB ( 32 feats) -> mejor iteración (early stopping): 516
ALL (154 feats) -> mejor iteración (early stopping): 133


## 4. Métricas de los tres, lado a lado

Métricas idénticas a `baseline.ipynb`: Cohen's Kappa y F1-macro vs *Expert* (target) y vs *Dreem* (2da anotación, descartando sus Unknown), `classification_report` y `confusion_matrix` a 5 clases, y la versión colapsada a 4 clases (Wake / Light=N1+N2 / Deep=N3 / REM).

In [5]:
from sklearn.metrics import f1_score, cohen_kappa_score, classification_report, confusion_matrix

map4 = {0: 0, 1: 1, 2: 1, 3: 2, 4: 3}
names4 = ['Wake', 'Light', 'Deep', 'REM']
remap = np.vectorize(map4.get)

def evaluate(name, clf):
    X_test = df[FEATURE_SETS[name]].iloc[test_pos]
    y_pred = clf.predict(X_test)

    kappa_exp = cohen_kappa_score(y_test, y_pred)
    f1_exp = f1_score(y_test, y_pred, average='macro')
    mask = dreem_test != 5
    kappa_dreem = cohen_kappa_score(dreem_test[mask], y_pred[mask])
    f1_dreem = f1_score(dreem_test[mask], y_pred[mask], average='macro')

    y_test4, y_pred4 = remap(y_test), remap(y_pred)
    kappa_exp4 = cohen_kappa_score(y_test4, y_pred4)
    f1_exp4 = f1_score(y_test4, y_pred4, average='macro')

    print(f"===== {name} ({len(FEATURE_SETS[name])} features) =====")
    print(f"Expert          ->  Kappa: {kappa_exp:.3f}   F1-macro: {f1_exp:.3f}")
    print(f"Dreem           ->  Kappa: {kappa_dreem:.3f}   F1-macro: {f1_dreem:.3f}")
    print(f"Expert (4 clases) ->  Kappa: {kappa_exp4:.3f}   F1-macro: {f1_exp4:.3f}")
    print("\nReporte por clase (vs Expert):")
    print(classification_report(y_test, y_pred, target_names=['Wake','N1','N2','N3','REM']))
    print("Matriz de confusión (vs Expert):")
    print(confusion_matrix(y_test, y_pred, labels=range(5)))
    print()
    return {'set': name, 'kappa_exp': kappa_exp, 'f1_exp': f1_exp,
            'kappa_dreem': kappa_dreem, 'f1_dreem': f1_dreem,
            'kappa_exp4': kappa_exp4, 'f1_exp4': f1_exp4}

results = [evaluate(name, models[name]) for name in FEATURE_SETS]

===== HC (122 features) =====
Expert          ->  Kappa: 0.378   F1-macro: 0.471
Dreem           ->  Kappa: 0.376   F1-macro: 0.466
Expert (4 clases) ->  Kappa: 0.387   F1-macro: 0.587

Reporte por clase (vs Expert):
              precision    recall  f1-score   support

        Wake       0.72      0.65      0.68      5655
          N1       0.25      0.01      0.01      3375
          N2       0.55      0.56      0.55     17365
          N3       0.64      0.50      0.56      9081
         REM       0.46      0.66      0.54     11951

    accuracy                           0.55     47427
   macro avg       0.52      0.48      0.47     47427
weighted avg       0.54      0.55      0.53     47427

Matriz de confusión (vs Expert):
[[3669   14  632  509  831]
 [ 455   22 1075  235 1588]
 [ 588   28 9705 1480 5564]
 [ 225    1 2823 4579 1453]
 [ 142   24 3463  395 7927]]

===== EMB (32 features) =====
Expert          ->  Kappa: 0.287   F1-macro: 0.397
Dreem           ->  Kappa: 0.291   F1-

### Tabla resumen + deltas

El aporte de los embeddings, en dos números: `EMB - HC` (¿los embeddings solos igualan a las handcrafted?) y `ALL - HC` (¿sumar los embeddings a las handcrafted mejora algo?).

In [6]:
summary = pd.DataFrame(results).set_index('set')
summary = summary.round(4)
print("Resumen (test set común a los tres):")
print(summary.to_string())

deltas = pd.DataFrame({
    'EMB - HC': summary.loc['EMB'] - summary.loc['HC'],
    'ALL - HC': summary.loc['ALL'] - summary.loc['HC'],
}).T.round(4)
print("\nDeltas vs HC (positivo = los embeddings ayudan):")
print(deltas.to_string())

Resumen (test set común a los tres):
     kappa_exp  f1_exp  kappa_dreem  f1_dreem  kappa_exp4  f1_exp4
set                                                               
HC      0.3776  0.4707       0.3762    0.4662      0.3865   0.5873
EMB     0.2867  0.3971       0.2909    0.3989      0.2919   0.4973
ALL     0.3762  0.4686       0.3749    0.4638      0.3847   0.5859

Deltas vs HC (positivo = los embeddings ayudan):
          kappa_exp  f1_exp  kappa_dreem  f1_dreem  kappa_exp4  f1_exp4
EMB - HC    -0.0909 -0.0736      -0.0853   -0.0673     -0.0946  -0.0900
ALL - HC    -0.0014 -0.0021      -0.0013   -0.0024     -0.0018  -0.0014


## 5. ¿XGBoost usa los embeddings o los ignora?

Sobre el modelo **FEAT_ALL** (que tuvo acceso a ambos grupos), miramos la `feature_importance` (`gain`): top 30 marcando `[EMB]` vs `[HC]`, más el gain **agregado** de cada grupo. Si el modelo apenas toca las `emb_*` cuando tiene las handcrafted disponibles, la ganancia agregada de los embeddings será marginal.

In [7]:
booster = models['ALL'].get_booster()
gains = booster.get_score(importance_type='gain')  # {feature_name: gain}
emb_set = set(emb_cols)

imp = (pd.Series(gains, name='gain').sort_values(ascending=False)
         .rename_axis('feature').reset_index())
imp['grupo'] = np.where(imp['feature'].isin(emb_set), 'EMB', 'HC')

print("Top 30 features por gain (modelo ALL):")
print(imp.head(30).to_string(index=False))

total_gain = imp['gain'].sum()
by_group = imp.groupby('grupo')['gain'].agg(['sum', 'count'])
by_group['frac_gain'] = (by_group['sum'] / total_gain).round(4)
# nota: 'count' = features con gain>0 (features nunca usadas por el modelo no aparecen)
print("\nGain agregado por grupo:")
print(by_group.to_string())
print(f"\nFeatures emb_* usadas por el modelo: {by_group.loc['EMB','count'] if 'EMB' in by_group.index else 0} / {len(emb_cols)}")
print(f"Fracción del gain total aportada por embeddings: "
      f"{by_group.loc['EMB','frac_gain'] if 'EMB' in by_group.index else 0.0:.1%}")

Top 30 features por gain (modelo ALL):
              feature       gain grupo
         hr_mean_rstd 108.557762    HC
           epoch_frac  83.049721    HC
       hr_median_rstd  63.034843    HC
immobility_frac_rmean  46.713821    HC
         hr_std_rmean  44.933151    HC
       enmo_mean_rstd  38.549065    HC
          hr_std_rstd  36.872608    HC
    epochs_since_move  31.650047    HC
        hr_slope_rstd  29.678741    HC
         acc_ptp_rstd  27.480766    HC
        jerk_std_lag2  26.895555    HC
        jerk_std_rstd  24.558134    HC
          hr_std_lag2  23.777336    HC
          hr_ptp_lag2  23.508764    HC
               emb_31  23.300898   EMB
         acc_std_lag2  23.024698    HC
         acc_ptp_lag2  22.495541    HC
         acc_std_rstd  21.964382    HC
         hr_ptp_rmean  20.202660    HC
         hr_mean_lag2  20.059719    HC
        enmo_std_rstd  19.990225    HC
          hr_ptp_rstd  19.396383    HC
       hr_median_lag2  19.041246    HC
             jerk_std  17